In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random

# Rastgelelik tohumunu sabitleyelim (Videodaki sonuçlarla birebir örtüşmesi için)
g = torch.Generator().manual_seed(2147483647)
random.seed(42)

print(f"PyTorch Versiyonu: {torch.__version__}")

PyTorch Versiyonu: 2.9.1


In [2]:
# Veriyi okuma
words = open('names.txt', 'r').read().splitlines()
print(f"Toplam isim sayısı: {len(words)}")

# Karakter kümesi ve mapping'ler
chars = sorted(list(set(''.join(words))))
stoi = {s: i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)
print(f"Vocab boyutu: {vocab_size}")

# Veri seti oluşturma fonksiyonu
block_size = 3 # Bağlam uzunluğu (context length)

def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr   = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte   = build_dataset(words[n2:])

Toplam isim sayısı: 32033
Vocab boyutu: 27
torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [3]:
# Elle hesapladığımız gradientler ile PyTorch autograd gradientlerini karşılaştıran yardımcı fonksiyon
def cmp(s, dt, t):
    if t.grad is None:
        print(f'{s:15s} | HATA: t.grad bulunamadı!')
        return
    ex = torch.all(dt == t.grad).item()
    app = torch.allclose(dt, t.grad, atol=1e-5)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

In [4]:
n_embd = 10    # Karakter gömme boyutu (embedding dimension)
n_hidden = 64  # Hidden layer nöron sayısı

g = torch.Generator().manual_seed(2147483647) # Birebir uyum için seed
C  = torch.randn((vocab_size, n_embd),            generator=g)
# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidden,                        generator=g) * 0.1
# Layer 2
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.1
b2 = torch.randn(vocab_size,                      generator=g) * 0.1
# BatchNorm parametreleri
bngain = torch.randn((1, n_hidden))*0.1 + 1.0
bnbias = torch.randn((1, n_hidden))*0.1

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print("Toplam parametre sayısı:", sum(p.nelement() for p in parameters))

for p in parameters:
    p.requires_grad = True

Toplam parametre sayısı: 4137


In [ ]:
batch_size = 32
n = batch_size # Kolaylık için kısa değişken adı

# Minibatch oluşturma
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix]

# Forward Pass 
emb = C[Xb] # (n, block_size, n_embd)
embcat = emb.view(emb.shape[0], -1) # (n, n_embd * block_size)

# Linear layer 1
hprebn = embcat @ W1 + b1 # (n, n_hidden)

# BatchNorm layer
bnmeani = 1/n * hprebn.sum(0, keepdim=True) # (1, n_hidden)
bndiff = hprebn - bnmeani # (n, n_hidden)
bndiff2 = bndiff**2 # (n, n_hidden)
bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # Bessel düzeltmesi (n-1 bölmesi) (1, n_hidden)
bnvar_inv = (bnvar + 1e-5)**-0.5 # (1, n_hidden)
bnraw = bndiff * bnvar_inv # (n, n_hidden)
hpreact = bngain * bnraw + bnbias # (n, n_hidden)

# Non-linearity
h = torch.tanh(hpreact) # (n, n_hidden)

# Linear layer 2
logits = h @ W2 + b2 # (n, vocab_size)

# Cross Entropy Loss (F.cross_entropy(logits, Yb) ile aynı)
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes # Sayısal kararlılık için max çıkarma
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdims=True)
counts_sum_inv = counts_sum**-1
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()

# PyTorch Backward Pass
for p in parameters:
    p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv,
          norm_logits, logit_maxes, logits, h, hpreact, bnraw,
          bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani,
          embcat, emb]:
    t.retain_grad()

loss.backward()
print(f"Loss değeri: {loss.item():.4f}")

Loss değeri: 3.3329


In [ ]:
# Egzersiz 1: Her ara değişkenin gradient'i elle yazıldı ve PyTorch ile doğrulandı.

# 1. dlogprobs: loss = -logprobs[range(n), Yb].mean()
dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), Yb] = -1.0 / n

# 2. dprobs: logprobs = probs.log()
dprobs = (1.0 / probs) * dlogprobs

# 3. dcounts_sum_inv: probs = counts * counts_sum_inv
dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True)

# 4. dcounts: probs = counts * counts_sum_inv (counts birinci kolu)
dcounts = counts_sum_inv * dprobs

# 5. dcounts_sum: counts_sum_inv = counts_sum**-1
dcounts_sum = (-counts_sum**-2) * dcounts_sum_inv
# dcounts ikince kolu: counts_sum = counts.sum(1, keepdims=True)
dcounts += torch.ones_like(counts) * dcounts_sum

# 6. dnorm_logits: counts = norm_logits.exp()
dnorm_logits = counts * dcounts

# 7. dlogit_maxes: norm_logits = logits - logit_maxes
dlogit_maxes = (-dnorm_logits).sum(1, keepdim=True)

# 8. dlogits: norm_logits = logits - logit_maxes VE logit_maxes = logits.max(1, keepdim=True).values
dlogits = dnorm_logits.clone()
dlogits += F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * dlogit_maxes

# 9. dh, dW2, db2: logits = h @ W2 + b2
dh = dlogits @ W2.T
dW2 = h.T @ dlogits
db2 = dlogits.sum(0)

# 10. dhpreact: h = torch.tanh(hpreact)
dhpreact = (1.0 - h**2) * dh

# 11. dbngain, dbnraw, dbnbias: hpreact = bngain * bnraw + bnbias
dbngain = (bnraw * dhpreact).sum(0, keepdim=True)
dbnraw = bngain * dhpreact
dbnbias = dhpreact.sum(0, keepdim=True)

# 12. dbndiff, dbnvar_inv: bnraw = bndiff * bnvar_inv
dbndiff = bnvar_inv * dbnraw
dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)

# 13. dbnvar: bnvar_inv = (bnvar + 1e-5)**-0.5
dbnvar = (-0.5 * (bnvar + 1e-5)**-1.5) * dbnvar_inv

# 14. dbndiff2: bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True)
dbndiff2 = (1.0 / (n - 1)) * torch.ones_like(bndiff2) * dbnvar
dbndiff += (2 * bndiff) * dbndiff2

# 15. dhprebn, dbnmeani: bndiff = hprebn - bnmeani
dhprebn = dbndiff.clone()
dbnmeani = (-dbndiff).sum(0)
dhprebn += 1.0 / n * (torch.ones_like(hprebn) * dbnmeani)

# 16. dembcat, dW1, db1: hprebn = embcat @ W1 + b1
dembcat = dhprebn @ W1.T
dW1 = embcat.T @ dhprebn
db1 = dhprebn.sum(0)

# 17. demb: embcat = emb.view(emb.shape[0], -1)
demb = dembcat.view(emb.shape)

# 18. dC: emb = C[Xb]
dC = torch.zeros_like(C)
for k in range(Xb.shape[0]):
    for j in range(Xb.shape[1]):
        ix = Xb[k, j]
        dC[ix] += demb[k, j]

# --- Karşılaştırmaları Çalıştır ---
print("--- EGZERSİZ 1 SONUÇLARI ---")
cmp('logprobs', dlogprobs, logprobs)
cmp('probs', dprobs, probs)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)
cmp('counts_sum', dcounts_sum, counts_sum)
cmp('counts', dcounts, counts)
cmp('norm_logits', dnorm_logits, norm_logits)
cmp('logit_maxes', dlogit_maxes, logit_maxes)
cmp('logits', dlogits, logits)
cmp('h', dh, h)
cmp('W2', dW2, W2)
cmp('b2', db2, b2)
cmp('hpreact', dhpreact, hpreact)
cmp('bngain', dbngain, bngain)
cmp('bnbias', dbnbias, bnbias)
cmp('bnraw', dbnraw, bnraw)
cmp('bnvar_inv', dbnvar_inv, bnvar_inv)
cmp('bnvar', dbnvar, bnvar)
cmp('bndiff2', dbndiff2, bndiff2)
cmp('bndiff', dbndiff, bndiff)
cmp('bnmeani', dbnmeani, bnmeani)
cmp('hprebn', dhprebn, hprebn)
cmp('embcat', dembcat, embcat)
cmp('W1', dW1, W1)
cmp('b1', db1, b1)
cmp('emb', demb, emb)
cmp('C', dC, C)

--- EGZERSİZ 1 SONUÇLARI ---
logprobs        | exact: True  | approximate: True  | maxdiff: 0.0
probs           | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0
counts          | exact: True  | approximate: True  | maxdiff: 0.0
norm_logits     | exact: True  | approximate: True  | maxdiff: 0.0
logit_maxes     | exact: True  | approximate: True  | maxdiff: 0.0
logits          | exact: True  | approximate: True  | maxdiff: 0.0
h               | exact: True  | approximate: True  | maxdiff: 0.0
W2              | exact: True  | approximate: True  | maxdiff: 0.0
b2              | exact: True  | approximate: True  | maxdiff: 0.0
hpreact         | exact: True  | approximate: True  | maxdiff: 0.0
bngain          | exact: True  | approximate: True  | maxdiff: 0.0
bnbias          | exact: True  | approximate: True  | maxdiff: 0.0
bnraw           | exact: True  | 